In [5]:
from pathlib import Path

dataset_root = Path(r"C:\Users\Shourya\Desktop\SAMPLE_dataset_public")

mat_files = list(dataset_root.rglob("*.mat"))

print("Total MAT files:", len(mat_files))

Total MAT files: 2690


In [6]:
import pandas as pd
from scipy.io import loadmat
from tqdm import tqdm

records = []

for file in tqdm(mat_files):

    data = loadmat(file)

    records.append({
        "filepath": str(file),
        "target_name": str(data["target_name"][0]),
        "azimuth": float(data["azimuth"][0][0]),
        "elevation": float(data["elevation"][0][0]),
        "is_synthetic": "synth" in str(file).lower()
    })

df = pd.DataFrame(records)

df.head()

100%|██████████| 2690/2690 [00:02<00:00, 1164.21it/s]


,filepath,target_name,azimuth,elevation,is_synthetic
0,C:\Users\Shourya\Desktop\SAMPLE_dataset_public...,2s1_gun,10.224838,15.015625,False
1,C:\Users\Shourya\Desktop\SAMPLE_dataset_public...,2s1_gun,11.224838,14.699219,False
2,C:\Users\Shourya\Desktop\SAMPLE_dataset_public...,2s1_gun,12.224838,14.992188,False
3,C:\Users\Shourya\Desktop\SAMPLE_dataset_public...,2s1_gun,13.224838,15.054688,False
4,C:\Users\Shourya\Desktop\SAMPLE_dataset_public...,2s1_gun,14.224838,15.140625,False


In [7]:
import pickle

df.to_pickle("sample_metadata.pkl")
df.to_csv("sample_metadata.pkl", index = False)

In [8]:
df["target_name"].value_counts()

target_name
m60_tank           352
2s1_gun            348
zsu23-4_gun        348
m1_tank            258
m35_truck          258
m548_transport     256
m2_tank            256
t72_tank           216
bmp2_tank          214
btr70_transport    184
Name: count, dtype: int64

In [9]:
target_type_map = {
    "m60_tank": "tank",
    "m1_tank": "tank",
    "m2_tank": "tank",
    "t72_tank": "tank",
    "bmp2_tank": "tank",
    "2s1_gun": "artillery",
    "zsu23-4_gun": "artillery",
    "m35_truck": "truck",
    "m548_transport": "transport",
    "btr70_transport": "transport"
}

mobility_map = {
    "m60_tank": "tracked",
    "m1_tank": "tracked",
    "m2_tank": "tracked",
    "t72_tank": "tracked",
    "bmp2_tank": "tracked",
    "2s1_gun": "tracked",
    "zsu23-4_gun": "tracked",
    "m35_truck": "wheeled",
    "m548_transport": "tracked",
    "btr70_transport": "wheeled"
}

In [10]:
df["target_type"] = df["target_name"].map(target_type_map)
df["mobility"] = df["target_name"].map(mobility_map)

In [11]:
df = df.drop(columns=["azimuth","elevation", "is_synthetic"])
df.to_pickle("sample_metadata.pkl")


In [12]:

import random
import pandas as pd

all_classes = sorted(df["target_name"].unique())
all_mob = sorted(df["mobility"].unique())

qa_records = []

for _, row in df.iterrows():

    true_class = row["target_name"]
    true_mob = row["mobility"]

    # ---------- Positive class ----------
    qa_records.append({
        "filepath": row["filepath"],
        "question": f"Is this a {true_class.replace('_', ' ')}?",
        "answer": "yes"
    })

    # ---------- Negative class ----------
    neg_class = random.choice(
        [c for c in all_classes if c != true_class]
    )

    qa_records.append({
        "filepath": row["filepath"],
        "question": f"Is this a {neg_class.replace('_', ' ')}?",
        "answer": "no"
    })

    # ---------- Positive mobility ----------
    qa_records.append({
        "filepath": row["filepath"],
        "question": f"Is this a {true_mob} vehicle?",
        "answer": "yes"
    })

    # ---------- Negative mobility ----------
    neg_mob = random.choice(
        [m for m in all_mob if m != true_mob]
    )

    qa_records.append({
        "filepath": row["filepath"],
        "question": f"Is this a {neg_mob} vehicle?",
        "answer": "no"
    })

qa_df = pd.DataFrame(qa_records)

print(qa_df["answer"].value_counts())

answer
yes    5380
no     5380
Name: count, dtype: int64


In [13]:
qa_df = qa_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [14]:
qa_df.to_pickle("sample_qa.pkl")

In [15]:
qa_df = pd.read_pickle("sample_qa.pkl")

In [16]:
qa_df.head()

,filepath,question,answer
0,C:\Users\Shourya\Desktop\SAMPLE_dataset_public...,Is this a zsu23-4 gun?,yes
1,C:\Users\Shourya\Desktop\SAMPLE_dataset_public...,Is this a m35 truck?,yes
2,C:\Users\Shourya\Desktop\SAMPLE_dataset_public...,Is this a zsu23-4 gun?,no
3,C:\Users\Shourya\Desktop\SAMPLE_dataset_public...,Is this a tracked vehicle?,yes
4,C:\Users\Shourya\Desktop\SAMPLE_dataset_public...,Is this a tracked vehicle?,yes


In [17]:
qa_df["filepath"].nunique()

2690

In [18]:
from sklearn.model_selection import train_test_split

unique_files = qa_df["filepath"].unique()

train_files, temp_files = train_test_split(
    unique_files,
    test_size=0.2,
    random_state=42
)

val_files, test_files = train_test_split(
    temp_files,
    test_size=0.5,
    random_state=42
)

print(len(train_files))
print(len(val_files))
print(len(test_files))

2152
269
269


In [19]:
train_df = qa_df[qa_df["filepath"].isin(train_files)].reset_index(drop=True)

val_df = qa_df[qa_df["filepath"].isin(val_files)].reset_index(drop=True)

test_df = qa_df[qa_df["filepath"].isin(test_files)].reset_index(drop=True)

print(len(train_df))
print(len(val_df))
print(len(test_df))

8608
1076
1076


In [20]:
from collections import Counter

counter = Counter()

for q in train_df["question"]:
    counter.update(q.lower().split())

word2idx_2 = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in counter:
    word2idx_2[word] = len(word2idx_2)

idx2word_2 = {
    v:k
    for k,v in word2idx_2.items()
}

print(word2idx_2)
print(len(word2idx_2))

{'<PAD>': 0, '<UNK>': 1, 'is': 2, 'this': 3, 'a': 4, 'zsu23-4': 5, 'gun?': 6, 'm35': 7, 'truck?': 8, 'tracked': 9, 'vehicle?': 10, 'm60': 11, 'tank?': 12, 'bmp2': 13, 'm1': 14, 'wheeled': 15, 'm548': 16, 'transport?': 17, 't72': 18, 'm2': 19, 'btr70': 20, '2s1': 21}
22


In [21]:
max_len = max(
    len(q.lower().split())
    for q in train_df["question"]
)

print(max_len)

5


In [22]:
def encode_question(question):

    words = question.lower().split()

    tokens = [
        word2idx_2.get(
            word,
            word2idx_2["<UNK>"]
        )
        for word in words
    ]

    if len(tokens) < max_len:

        tokens += (
            [word2idx_2["<PAD>"]]
            * (max_len - len(tokens))
        )

    return tokens[:max_len]

In [23]:
train_processed = []

for _, row in train_df.iterrows():

    train_processed.append({

        "filepath": row["filepath"],

        "question_tokens":
            encode_question(
                row["question"]
            ),

        "answer":
            1 if row["answer"] == "yes"
            else 0
    })

In [24]:
val_processed = []

for _, row in val_df.iterrows():

    val_processed.append({

        "filepath": row["filepath"],

        "question_tokens":
            encode_question(
                row["question"]
            ),

        "answer":
            1 if row["answer"] == "yes"
            else 0
    })

In [25]:
test_processed = []

for _, row in test_df.iterrows():

    test_processed.append({

        "filepath": row["filepath"],

        "question_tokens":
            encode_question(
                row["question"]
            ),

        "answer":
            1 if row["answer"] == "yes"
            else 0
    })

In [26]:
import pickle

with open("data/train_processed.pkl", "wb") as f:
    pickle.dump(train_processed, f)

with open("data/val_processed.pkl", "wb") as f:
    pickle.dump(val_processed, f)

with open("data/test_processed.pkl", "wb") as f:
    pickle.dump(test_processed, f)

with open("data/word2idx_2.pkl", "wb") as f:
    pickle.dump(word2idx_2, f)

with open("data/idx2word_2.pkl", "wb") as f:
    pickle.dump(idx2word_2, f)

In [27]:
print(train_processed[0])

{'filepath': 'C:\\Users\\Shourya\\Desktop\\SAMPLE_dataset_public\\SAMPLE_dataset_public\\mat_files\\synth\\zsu23\\zsu23_synth_A_elevDeg_015_azCenter_014_99_serial_d08.mat', 'question_tokens': [2, 3, 4, 5, 6], 'answer': 1}
